# 08.02 · Document parsing & chunking

**Chapter:** Retrieval-Augmented Generation · **Milestone:** M3

## Goals
- [ ] Parse a real PDF with three tools and compare what survives (headings, tables, reading order)
- [ ] Chunk with four strategies: fixed-size, recursive, structure-aware (by headings), semantic
- [ ] Know the trade-offs: chunk size vs precision vs context

## Resources
- [Docling](https://github.com/docling-project/docling) · [MarkItDown](https://github.com/microsoft/markitdown) · [PyMuPDF](https://pymupdf.readthedocs.io/) — free
- [Pinecone – Chunking strategies](https://www.pinecone.io/learn/chunking-strategies/) — free
- [Chroma – Evaluating chunking](https://research.trychroma.com/evaluating-chunking) — free

| Parser | Strengths | Cost |
|---|---|---|
| **PyMuPDF** (`pymupdf`) | very fast text extraction; blocks with positions | no structure, tables as loose text |
| **MarkItDown** | anything → Markdown (PDF, DOCX, PPTX, XLSX, HTML) | PDF via plain text extraction |
| **Docling** | layout model: headings, tables, reading order, figures → Markdown/JSON | slower, downloads models |
| Unstructured | many formats, element types | heavier |
| Paid: LlamaParse, Reducto, Azure Document Intelligence, Google Document AI, Mistral OCR | best on messy scans/forms | per page |
| VLM parsing (Gemini, GPT, Claude, olmOCR, SmolDocling) | page images → Markdown | per page, can hallucinate |

In [ ]:
import re
import time
import urllib.request

import numpy as np
import pymupdf
import tiktoken

from aieng import DATA

enc = tiktoken.get_encoding("o200k_base")
ntok = lambda s: len(enc.encode(s))

## 1. A real PDF: "Attention Is All You Need"
Equations, tables, figures — a realistic test. We keep the first 8 pages (through Table 2) to stay fast.

In [ ]:
full = DATA / "attention.pdf"
if not full.exists():
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", full)
pdf_path = DATA / "attention-p1-8.pdf"
src = pymupdf.open(full)
out = pymupdf.open(); out.insert_pdf(src, from_page=0, to_page=7); out.save(pdf_path)
print(src.page_count, "pages in full paper; using 8")

## 2. Three parsers

In [ ]:
parsed = {}

start = time.perf_counter()
parsed["pymupdf"] = "\n".join(page.get_text() for page in pymupdf.open(pdf_path))
print(f"pymupdf    {time.perf_counter() - start:5.1f}s")

from markitdown import MarkItDown

start = time.perf_counter()
parsed["markitdown"] = MarkItDown().convert(str(pdf_path)).text_content
print(f"markitdown {time.perf_counter() - start:5.1f}s")

from docling.document_converter import DocumentConverter

start = time.perf_counter()
parsed["docling"] = DocumentConverter().convert(pdf_path).document.export_to_markdown()
print(f"docling    {time.perf_counter() - start:5.1f}s (first run downloads layout models)")

for name, text in parsed.items():
    headings = len(re.findall(r"^#+ ", text, flags=re.M))
    table_rows = len(re.findall(r"^\|.*\|$", text, flags=re.M))
    print(f"{name:11} {len(text):6} chars | markdown headings: {headings:2} | table rows: {table_rows}")

In [ ]:
i = parsed["docling"].find("Table 1:")
print(parsed["docling"][i : i + 900])

Docling recovers section headings and the table as a Markdown table; plain extractors give a flat stream where table cells
become disconnected numbers. If your users ask about tables, the parser decides whether RAG can ever answer.

## 3. Chunking strategies
Why chunk at all? Embeddings of long texts blur (07.01); retrieval granularity should match question granularity;
and the context budget is finite (06.04). Typical sizes: 200–800 tokens with 10–20% overlap — then **measure**.

In [ ]:
text = parsed["docling"]


def fixed_size(text, size=300, overlap=50):
    toks = enc.encode(text)
    return [enc.decode(toks[i : i + size]) for i in range(0, len(toks), size - overlap)]


from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name="o200k_base", chunk_size=300, chunk_overlap=50)


def by_headings(markdown, max_tokens=500):
    """Structure-aware: split on Markdown headings, keep the heading path as context, sub-split long sections."""
    sections = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2"), ("###", "h3")]).split_text(markdown)
    chunks = []
    for s in sections:
        header = " > ".join(s.metadata.values())
        for piece in (recursive.split_text(s.page_content) if ntok(s.page_content) > max_tokens else [s.page_content]):
            chunks.append(f"[{header}]\n{piece}")
    return chunks

**Semantic chunking**: embed sentences, start a new chunk where similarity between neighbours drops (a topic shift).

In [ ]:
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("BAAI/bge-small-en-v1.5")


def semantic(text, percentile=15, max_tokens=500):
    sents = [s for s in re.split(r"(?<=[.!?])\s+|\n{2,}", text) if s.strip()]
    E = emb.encode(sents, normalize_embeddings=True)
    sims = (E[:-1] * E[1:]).sum(1)
    cut = np.percentile(sims, percentile)
    chunks, cur = [], [sents[0]]
    for s, sim in zip(sents[1:], sims):
        if sim < cut or ntok(" ".join(cur + [s])) > max_tokens:
            chunks.append(" ".join(cur)); cur = []
        cur.append(s)
    return chunks + [" ".join(cur)]


strategies = {"fixed 300/50": fixed_size(text), "recursive 300/50": recursive.split_text(text),
              "by headings": by_headings(text), "semantic": semantic(text)}
for name, chunks in strategies.items():
    sizes = [ntok(c) for c in chunks]
    print(f"{name:17} {len(chunks):3} chunks | tokens median {int(np.median(sizes)):4} min {min(sizes):4} max {max(sizes):4}")

print("\n--- a heading-aware chunk:\n", next(ch for ch in strategies["by headings"] if "Scaled Dot-Product" in ch)[:400])

## 4. Which one retrieves best?
Write questions about the paper, find which chunk would answer them, and compare strategies by whether the top-3 chunks contain a key phrase.

In [ ]:
QUESTIONS = [  # (question, phrase that must be in a retrieved chunk)
    ("How many layers does the encoder have?", "N = 6"),
    ("What is the dimension of the model's outputs (d_model)?", "512"),
    ("Why divide the dot products by the square root of d_k?", "softmax"),
    ("Which optimizer was used for training?", "Adam"),
    ("What is the complexity per layer of self-attention?", "O(n"),
    ("What does multi-head attention allow the model to do?", "different representation"),
]


def top_k(chunks, question, k=3):
    C = emb.encode(chunks, normalize_embeddings=True)
    q = emb.encode(["Represent this sentence for searching relevant passages: " + question], normalize_embeddings=True)[0]
    return [chunks[i] for i in np.argsort(-(C @ q))[:k]]


for name, chunks in strategies.items():
    hits = sum(any(phrase in c for c in top_k(chunks, q)) for q, phrase in QUESTIONS)
    print(f"{name:17} {hits}/{len(QUESTIONS)}")

With 6 questions this is anecdote, not evidence — but it's the right *method*. Takeaways from larger studies: structure-aware chunking
with heading context is a strong default; semantic chunking is rarely worth its cost; chunk size matters more than strategy.
When a question fails, check ingestion first: is the answer even in the parsed text, or was it lost in a table/figure/page cut?

## Exercises
1. Parse the **full** paper with Docling and add 5 questions about the results tables. Then try a scanned PDF (photo of a page): which parsers survive?
2. Sweep fixed-size chunk sizes (100, 200, 400, 800 tokens) on 15 questions you write. Plot hit rate.
3. Add the paper title + section path to every chunk before embedding (cheap "contextual" chunks) and re-measure. (Full contextual retrieval: 08.04.)

In [ ]:
# your code here